There's been a lot of [controversy](https://www.nytimes.com/2007/03/18/opinion/18iht-edsafire.4943390.html#:~:text=Both%20words%20can%20function%20as,woman%20is%20what%20the%20O.E.D.) surrounding the word "female" as a noun (with some going as far as to claim it shouldn't even be use as an adjective when applied to humans). Anecdotally, I've noticed quite a few people use "female" as a noun in the same breath they say "man" (as opposed to "male") in the same sentence, which does rub me the wrong way, given that you rarely ever hear someone refer to women as "women" but men as "males". Is this confirmation bias though? Time to analyze a [million reddit comments](https://www.kaggle.com/smagnan/1-million-reddit-comments-from-40-subreddits).

First, lets import our data and check our first 5 rows:

In [197]:
import pandas
import csv

df = pandas.read_csv("redditcommentsquoted.csv", delimiter=',', encoding='utf-8')
print(df.head())

       subreddit                                               body  \
0  gameofthrones  Your submission has been automatically removed...   
1            aww  Dont squeeze her with you massive hand, you me...   
2         gaming  It's pretty well known and it was a paid produ...   
3           news  You know we have laws against that currently c...   
4       politics  Yes, there is a difference between gentle supp...   

   controversiality  score  
0                 0      1  
1                 0     19  
2                 0      3  
3                 0     10  
4                 0      1  


Now, for some distinctions: I personally don't find "female" as an adjective to be problematic. "The female/male librarian" is fine, although some have called for us to change our language to say "The woman/man librarian". Anyway, this means that we need to determine if the word being used in the sentence is an adjective or a noun.

For this we'll import the Natural Language Toolkit and download packages that allow us to tag words with their function in a sentence.

In [171]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Melanie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Melanie\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

As an example, here you can see how it classifies the first sentence's use of female as an adjective (`JJ`), and the second sentence's use of female as a noun (`NN`)

In [172]:
female_as_adjective = "The female librarian walked to the desk."
female_as_verb = "I was talking to a female recently."

print(list(filter(lambda x: x[0] == 'female', nltk.pos_tag(nltk.word_tokenize(female_as_adjective)))))
print(list(filter(lambda x: x[0] == 'female', nltk.pos_tag(nltk.word_tokenize(female_as_verb)))))

[('female', 'JJ')]
[('female', 'NN')]


Next, we apply it to our dataset.

First, lets find all comments that mention words we're interested in. We only use comments that mention both genders. Comments with only one gendered word don't actually tell us anything about if the person uses "female/man" or "male/woman" combinations, because they could just be using "female/male" combinations.

In [184]:
group1 = ["female", "females", "woman", "women"]
group2 = ["male", "males", "man", "men"]

# Narrow down df to only comments that contain words we're interested in
df = df[df['body'].str.contains("females?|males?|wom[ae]n|m[ae]n|girls?|boys?")]

print("Identified {} comments with words we're interested in".format(df.shape[0]))

print("Comments processed: ")

comments_of_interest = []
for i, value in df.iterrows():
    
    bothFound = [False, False]
    # We use nltk instead of split() because words frequently have punctuation and other things attached
    for word in [x[0] for x in nltk.pos_tag(nltk.word_tokenize(value.body.lower()))]:
        if word in group1:
            bothFound[0] = True
            
        elif word in group2:
            bothFound[1] = True
        
        if bothFound == [True, True]:
            comments_of_interest.append(value)
            break
    
    # Just as a progress bar
    if i % 1000 == 0:
        print(i, end=" ")

print("Done!")

(1000000, 4)
(157392, 4)
Comments processed: 
4000 15000 18000 26000 27000 28000 36000 49000 56000 61000 65000 75000 76000 79000 81000 82000 108000 111000 114000 118000 119000 126000 127000 142000 147000 149000 150000 189000 201000 202000 210000 212000 213000 217000 218000 226000 232000 246000 251000 258000 259000 260000 262000 264000 274000 304000 309000 319000 323000 342000 343000 352000 362000 363000 373000 380000 381000 390000 398000 402000 417000 418000 419000 425000 428000 434000 435000 443000 444000 445000 446000 456000 467000 469000 470000 473000 482000 483000 490000 499000 501000 511000 513000 514000 515000 516000 524000 533000 541000 545000 546000 549000 554000 556000 570000 572000 573000 575000 578000 581000 584000 601000 602000 603000 612000 614000 615000 627000 631000 650000 651000 655000 671000 674000 677000 687000 695000 702000 703000 705000 707000 708000 709000 714000 720000 721000 727000 736000 737000 738000 743000 748000 753000 770000 776000 791000 792000 796000 81700

Next, we classify the breakdown of the words in each comment.

In [223]:
comment_classifications = []
words_of_interest = group1 + group2

for comment in comments_of_interest:
    comment_class = {}
    comment_class["body"] = comment.body
    
    tagged_words = nltk.pos_tag(nltk.word_tokenize(comment.body.lower()))
    
    for word, tag in tagged_words:
        if word in words_of_interest:
            # There's multiple types of nouns, so we condense into one.
            if tag == "NN" or tag == "NNS" or tag == "NNP":
                tag = "noun"
            elif tag == "JJ":
                tag = "adj"
            # This seems to occur when people use the phrase "male to female" or "female to male"
            elif tag == "VB" or tag == "VBP" or tag == "VBZ" or tag == "VBN":
                tag = "verb"
                
            # We want these to register as the same
            if word == "females":
                word = "female"
            elif word == "males":
                word = "male"
            elif word == "women":
                word = "woman"
            elif word == "men":
                word = "man"

            if (word, tag) in comment_class:
                comment_class[(word, tag)] += 1
            else:
                comment_class[(word, tag)] = 1

    comment_classifications.append(comment_class)

print(comment_classifications[:5])

[{'body': 'So, so many men do not understand this. You can SMELL the preying vibes coming off them when they approach you and try super super hard to impress you. It is super tiring and boring to talk to. I will happily chat to anyone, literally anyone, if they\'re just chill and being themselves and having a good time.\n\nThe other thing is that sometimes men try this, and even though they get better at conversations etc, they still think it\'s unfair or something when they don\'t get a girlfriend in like a week. It isn\'t about being "friendzoned" because you\'re the worst, it\'s the fact that not everyone will be your perfect bloody match. Think about how many people you meet and either think are just okay, or don\'t like that much. Then apply that same statistic to finding a partner, and heighten it a bit because you have to REALLY like someone and click with them to want to put your face close to their face (maybe unless drunk idk). It is NORMAL to struggle to find a good partner,

Let's see how many comments use female as a noun.

In [206]:
broad_classification = {}

for comment_class in comment_classifications:
    for key in comment_class:
        if key in broad_classification:
            broad_classification[key][0] += 1
            broad_classification[key][1] += comment_class[key]
        else:
            broad_classification[key] = [1, 1] # num comments, num usages

print("{} comments used 'female' as a noun, with {} total usages".format(broad_classification[("female", "noun")][0], broad_classification[("female", "noun")][1]))
print("{} comments used 'female' as an adjective, with {} total usages".format(broad_classification[("female", "adj")][0], broad_classification[("female", "adj")][1]))
print("{} comments used 'male' as a noun, with {} total usages".format(broad_classification[("male", "noun")][0], broad_classification[("male", "noun")][1]))
print("{} comments used 'male' as an adjective, with {} total usages".format(broad_classification[("male", "adj")][0], broad_classification[("male", "adj")][1]))

468 comments used 'female' as a noun, with 540 total usages
631 comments used 'female' as an adjective, with 771 total usages
721 comments used 'male' as a noun, with 878 total usages
423 comments used 'male' as an adjective, with 495 total usages


Okay, so how are people using these words in combination?

In [225]:
female_man_comb = 0
male_woman_comb = 0
woman_man_comb = 0
female_male_comb = 0

for comment_class in comment_classifications:
    comment_class_no_body = comment_class.copy()
    comment_class_no_body.pop('body', None)
    
    usages = [(word + tag) for word, tag in comment_class_no_body]
    combinations = [(a, b) for a in usages for b in usages]
    
    if ('femalenoun', 'mannoun') in combinations:
        female_man_comb += 1
    if ('malenoun', 'womannoun') in combinations:
        male_woman_comb += 1
    if ('womannoun', 'mannoun') in combinations:
        woman_man_comb += 1
    if ('femalenoun', 'malenoun') in combinations:
        female_male_comb += 1

print(female_man_comb)
print(male_woman_comb)
print(woman_man_comb)
print(female_male_comb)

164
352
2649
314


Here's a sentence from the data: 
> "Sorry women are not somehow superior on the [...]. Plenty of boys are [...]. This isn’t a male or female issue this is just a [...] issue."

However, this registers as both `male_woman_comb` and `female_man_comb`. So let's make it so that it looks for _unique_ instances.

In [232]:
female_man_comb = 0
male_woman_comb = 0

for comment_class in comment_classifications:
    comment_class_no_body = comment_class.copy()
    comment_class_no_body.pop('body', None)
    
    usages = [(word + tag) for word, tag in comment_class_no_body]
    combinations = [(a, b) for a in usages for b in usages]
    
    if ('femalenoun', 'mannoun') in combinations and ('malenoun', 'womannoun') not in combinations and ('womannoun', 'mannoun') not in combinations and ('femalenoun', 'malenoun') not in combinations:
        female_man_comb += 1
    if ('malenoun', 'womannoun') in combinations and ('femalenoun', 'mannoun') not in combinations and ('womannoun', 'mannoun') not in combinations and ('femalenoun', 'malenoun') not in combinations:
        male_woman_comb += 1

print(female_man_comb)
print(male_woman_comb)


58
150
